# Stable Diffusion Img2Img Inference - Kaggle

Notebook untuk inference Stable Diffusion dengan checkpoint safetensor dari Civit AI.

## Input yang diperlukan:

| Input | Deskripsi |
|-------|-----------|
| **CIVITAI_MODEL_URL** | URL model dari Civit AI. Bisa berupa: (1) Direct download: `https://civitai.com/api/download/models/VERSION_ID`, atau (2) Model page: `https://civitai.com/models/MODEL_ID/nama-model` |
| **CIVITAI_API_KEY** | API key dari [Civitai Account Settings](https://civitai.com/user/account). Diperlukan untuk model gated/private. Bisa dikosongkan untuk model publik. |
| **INPUT_IMAGE_PATH** | Path ke gambar (di Kaggle dataset) atau URL gambar |
| **PROMPT** | Prompt teks untuk generasi |

## Cara pakai di Kaggle:
1. Aktifkan **GPU** (Settings → Accelerator → GPU)
2. Tambah dataset berisi gambar input, atau gunakan URL gambar
3. Isi konfigurasi di Cell 2
4. Run semua cell

## 1. Setup & Install Dependencies

In [ ]:
# Kaggle: Pastikan GPU diaktifkan (Settings -> Accelerator -> GPU)
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
import os
import re
import requests
from pathlib import Path
from PIL import Image
import torch

from diffusers import StableDiffusionImg2ImgPipeline

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Konfigurasi Input

In [ ]:
# ============ KONFIGURASI INPUT ============

# Civit AI - 2 input utama
# https://civitai.com/api/download/models/2681234?type=Model&format=SafeTensor&size=pruned&fp=fp16
CIVITAI_MODEL_URL = "https://civitai.com/api/download/models/2681234?type=Model&format=SafeTensor&size=pruned&fp=fp16"  # URL download langsung ATAU model page: https://civitai.com/models/12345/modelName
CIVITAI_API_KEY = "ec89fac87e2094a076d43157e4fec4ac"  # API key dari https://civitai.com/user/account

# Input gambar dan prompt
INPUT_IMAGE_PATH = "/kaggle/input/your-dataset/your_image.png"  # Path ke gambar di Kaggle dataset ATAU URL gambar
PROMPT = "a beautiful landscape, highly detailed, 8k"
NEGATIVE_PROMPT = "blurry, low quality, distorted"

# Template prompt (format list) - untuk web Flask, pilih salah satu untuk isi otomatis
PROMPT_TEMPLATES = [
    "a beautiful flower, highly detailed, 8k",
    "a cute cat, photorealistic, soft lighting",
    "a serene landscape, mountains, sunset, 4k",
    "portrait of a person, professional photo, studio lighting",
    "abstract art, vibrant colors, modern design",
]

# Parameter inference
STRENGTH = 0.75  # 0.0-1.0, seberapa banyak mengubah gambar (0.75 = perubahan moderat)
GUIDANCE_SCALE = 7.5
NUM_INFERENCE_STEPS = 50
SEED = 42

## 3. Download Model dari Civit AI

In [ ]:
def get_civitai_download_url(url: str, api_key: str) -> str:
    """
    Mendapatkan URL download dari Civit AI.
    Mendukung:
    - Direct download URL: https://civitai.com/api/download/models/12345
    - Model page URL: https://civitai.com/models/12345/modelName
    """
    # Jika sudah direct download URL
    if "/api/download/models/" in url:
        return url.strip()
    
    # Jika model page URL, ekstrak model ID dan ambil download URL via API
    match = re.search(r'civitai\.com/models/(\d+)', url)
    if not match:
        raise ValueError(f"URL tidak valid. Gunakan format: https://civitai.com/models/MODEL_ID atau https://civitai.com/api/download/models/VERSION_ID")
    
    model_id = match.group(1)
    api_url = f"https://civitai.com/api/v1/models/{model_id}"
    
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    
    resp = requests.get(api_url, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    
    # Ambil primary file dari versi terbaru
    model_versions = data.get("modelVersions", [])
    if not model_versions:
        raise ValueError("Model tidak memiliki versi yang tersedia")
    
    # Prioritas: 1) Primary SafeTensor, 2) SafeTensor, 3) Primary, 4) First file
    for version in model_versions:
        files = version.get("files", [])
        primary_safetensor = next((f for f in files if f.get("primary") and f.get("metadata", {}).get("format") == "SafeTensor"), None)
        if primary_safetensor:
            return primary_safetensor.get("downloadUrl")
        safetensor = next((f for f in files if f.get("metadata", {}).get("format") == "SafeTensor"), None)
        if safetensor:
            return safetensor.get("downloadUrl")
        primary = next((f for f in files if f.get("primary")), None)
        if primary:
            return primary.get("downloadUrl")
        if files:
            return files[0].get("downloadUrl")
    
    # Fallback: download URL dari versi pertama
    first_version = model_versions[0]
    files = first_version.get("files", [])
    if files:
        return files[0].get("downloadUrl") or f"https://civitai.com/api/download/models/{first_version['id']}"
    
    return f"https://civitai.com/api/download/models/{first_version['id']}"


def download_civitai_model(url: str, api_key: str, save_dir: str = "/kaggle/working/models") -> str:
    """Download model dari Civit AI dan simpan sebagai safetensors."""
    download_url = get_civitai_download_url(url, api_key)
    
    # Tambahkan token ke URL jika ada API key
    if api_key:
        sep = "&" if "?" in download_url else "?"
        download_url = f"{download_url}{sep}token={api_key}"
    
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    resp = requests.get(download_url, headers=headers, stream=True)
    resp.raise_for_status()
    
    # Ambil filename dari Content-Disposition atau gunakan default
    content_disp = resp.headers.get("Content-Disposition")
    if content_disp and "filename=" in content_disp:
        filename = re.findall(r'filename[*]?=(?:UTF-8\'\')?["\']?([^"\';\n]+)', content_disp)
        filename = filename[0].strip() if filename else "model.safetensors"
    else:
        filename = "model.safetensors"
    
    save_path = os.path.join(save_dir, filename)
    
    print(f"Downloading model ke {save_path}...")
    with open(save_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download selesai!")
    return save_path

In [ ]:
# Download model dari Civit AI
MODEL_SAVE_DIR = "/kaggle/working/models"
model_path = download_civitai_model(CIVITAI_MODEL_URL, CIVITAI_API_KEY, MODEL_SAVE_DIR)

## 4. Load Checkpoint Safetensor & Pipeline

In [ ]:
# Load pipeline dari single file (safetensors atau ckpt)
pipe = StableDiffusionImg2ImgPipeline.from_single_file(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    use_safetensors=model_path.endswith(".safetensors"),
)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
pipe.set_progress_bar_config(disable=False)
print("Pipeline loaded successfully!")

## 5. Load Gambar Input

In [ ]:
def load_input_image(path_or_url: str) -> Image.Image:
    """Load gambar dari path file atau URL."""
    if path_or_url.startswith("http"):
        return Image.open(requests.get(path_or_url, stream=True).raw).convert("RGB")
    return Image.open(path_or_url).convert("RGB")


init_image = load_input_image(INPUT_IMAGE_PATH)
# Resize ke kelipatan 8 (required oleh SD)
w, h = init_image.size
w = (w // 8) * 8
h = (h // 8) * 8
init_image = init_image.resize((w, h), Image.Resampling.LANCZOS)
init_image

## 6. Inference Img2Img

In [ ]:
generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(SEED)

result = pipe(
    prompt=PROMPT,
    image=init_image,
    strength=STRENGTH,
    guidance_scale=GUIDANCE_SCALE,
    num_inference_steps=NUM_INFERENCE_STEPS,
    negative_prompt=NEGATIVE_PROMPT,
    generator=generator,
)

output_image = result.images[0]
output_image

In [ ]:
# Simpan hasil
output_path = "/kaggle/working/output.png"
output_image.save(output_path)
print(f"Hasil disimpan ke {output_path}")